# Architecture Design Council
**LLM agents debate a design brief and produce objective weights for a Grasshopper optimisation algorithm.**

### Flow
```
Cell 1 — Install & imports
Cell 2 — Config: API key, model, output path
Cell 3 — Document parsers (PDF + Excel)
Cell 4 — BaseAgent class
Cell 5 — Four specialist agents
Cell 6 — chairperson Agent
Cell 7 — Council orchestrator (Stage 1 → Stage 2 → chairperson)
Cell 8 — Run the council (streaming transcript)
Cell 9 — Inspect output + write weights_output.json
```

---
## Cell 1 — Install & imports

In [ ]:
# !pip install anthropic pymupdf openpyxl ipywidgets openai ipywidgets

In [ ]:
# import subprocess, sys
# subprocess.check_call([sys.executable, "-m", "pip", "install", 
#                        "anthropic", "pymupdf", "openpyxl"])

In [ ]:
import anthropic
import asyncio
import json
import os
import re
from pathlib import Path
from datetime import datetime, timezone
from typing import Optional
from IPython.display import display, Markdown

import fitz          # PyMuPDF
import openpyxl
from IPython.display import display, Markdown, clear_output

print("✓ imports ok")

---
## Cell 2 — Config

In [ ]:
import openpyxl, re, json, os
from pathlib import Path
from datetime import datetime, timezone
from openai import OpenAI
from IPython.display import display, Markdown

# ── LM Studio config ──────────────────────────────────────────────────────────
LM_STUDIO_BASE_URL = "http://localhost/v1"
LM_STUDIO_API_KEY  = "lm-studio"
MODEL              = "gpt-oss:20b"
MAX_TOKENS         = 5000

# ── Parameter definitions (injected into every agent's system prompt) ─────────
PARAMETER_DEFINITIONS = """
You are reasoning about three specific design parameters. Use these exact definitions:

1. COST (£ — British Sterling)
   The total financial expenditure required for the project or specific elements
   (facade vs. floor area), expressed in GBP. This represents benchmarked capital
   cost for construction. You have both min/max floor plate costs (£/m²) and
   facade costs split by type (glass vs. solid £/m²).
   Your weight for this parameter controls how strongly the optimiser penalises
   solutions that push toward the upper cost boundary.

2. CARBON EQUIVALENT (kg CO₂e/m³)
   A standard unit measuring carbon footprints, expressing the impact of greenhouse
   gases in terms of equivalent CO₂ warming. In this project it refers to Embodied
   Carbon — emissions from manufacturing, transporting, and installing building
   materials. Each structural material has a distinct carbon cost:
   concrete, timber, and steel all differ significantly.
   Your weight for this parameter controls how strongly the optimiser favours
   low-embodied-carbon material and facade choices.

3. GLAZING (% of facade area)
   The percentage of the total outward-facing facade area made up of transparent
   glazing (windows/curtain walls) versus opaque solid wall. This is a critical
   lever for both daylighting performance (BREEAM rating) and facade cost per m²
   (glass at £1,000/m² vs. solid at £500/m²). Higher glazing improves daylight
   but increases cost and may increase heat gain/loss.
   Your weight for this parameter controls how strongly the optimiser prioritises
   glazing ratio within the permitted range.
""".strip()

client = OpenAI(base_url=LM_STUDIO_BASE_URL, api_key=LM_STUDIO_API_KEY)

# ── Input file paths — edit these directly ────────────────────────────────────
INPUT_FILES = {
    "design_brief":         Path(r"/Input/Client Brief.xlsx"),
    "cost_consultancy":     Path(r"/Input/Costing Information.xlsx"),
    "sustainability":       Path(r"/Input/Sustainability.xlsx"),
    "building_regulations": Path(r"/Input/Building Regulations.xlsx"),
}

OUTPUT_PATH = Path(r"/Output/weights_output.json")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

def test_connection():
    try:
        models = client.models.list()
        available = [m.id for m in models.data]
        print(f"✓ LM Studio connected at {LM_STUDIO_BASE_URL}")
        print(f"  Available models: {available}")
        global MODEL
        if available and MODEL == "local-model":
            MODEL = available[0]
            print(f"  → Using: {MODEL}")
    except Exception as e:
        print(f"✗ Could not connect to LM Studio: {e}")

test_connection()

# ── Read any Excel file into a flat dict (handles ranges as min/max pairs) ────
def read_excel_inputs(path: Path) -> dict:
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb.active
    inputs = {}
    for row in ws.iter_rows(values_only=True):
        if not any(c is not None for c in row):
            continue
        key = str(row[0]).strip() if row[0] is not None else None
        val = row[1] if len(row) > 1 else None
        unit = row[2] if len(row) > 2 else None
        if key is None:
            continue
        try:
            inputs[key] = float(val)
        except (TypeError, ValueError):
            inputs[key] = str(val).strip() if val is not None else ""
        if unit is not None:
            inputs[f"{key}__unit"] = str(unit).strip()
    return inputs

# ── Load all input files ───────────────────────────────────────────────────────
raw = {name: read_excel_inputs(path) for name, path in INPUT_FILES.items()}

# ── Structured PROJECT_INPUTS pulled from Client_Brief ───────────────────────
cb = raw["design_brief"]
ci = raw["cost_consultancy"]
su = raw["sustainability"]
br = raw["building_regulations"]

PROJECT_INPUTS = {
    # From Client Brief
    "project_type":       cb.get("project_type", "Commercial Office"),
    "min_area_m2":        cb.get("min_area",   0.0),
    "max_area_m2":        cb.get("max_area",   0.0),
    "min_budget":         cb.get("min_budget", 0.0),
    "max_budget":         cb.get("max_budget", 0.0),

    # From Costing Information
    "min_cost_floor_plate":  ci.get("min_cost_floor_plate", 0.0),
    "max_cost_floor_plate":  ci.get("max_cost_floor_plate", 0.0),
    "cost_glass_facade":     ci.get("cost_glass_facade",    0.0),
    "cost_solid_facade":     ci.get("cost_solid_facade",    0.0),
    "material_cost_concrete":ci.get("material_cost_concrete", 0.0),
    "material_cost_timber":  ci.get("material_cost_timber",   0.0),
    "material_cost_steel":   ci.get("material_cost_steel",    0.0),

    # From Sustainability
    "min_glazing_ratio":          su.get("min_glazing_ratio", 0.0),
    "max_glazing_ratio":          su.get("max_glazing_ratio", 0.0),
    "concrete_embodied_carbon":   su.get("concrete_embodied_carbon", 0.0),
    "timber_embodied_carbon":     su.get("timber_embodied_carbon",   0.0),
    "steel_embodied_carbon":      su.get("steel_embodied_carbon",    0.0),

    # From Building Regulations
    "min_floor_height_m":  br.get("min_floor height", 0.0),
    "max_floor_height_m":  br.get("max_floor height", 0.0),
    "min_occupancy_per_m2":br.get("min_occupancy",    0.0),
    "max_occupancy_per_m2":br.get("max_occupancy",    0.0),
}

# ── Computed mid-point estimates (for cost agent reasoning) ───────────────────
PROJECT_INPUTS["mid_area_m2"]         = (PROJECT_INPUTS["min_area_m2"] + PROJECT_INPUTS["max_area_m2"]) / 2
PROJECT_INPUTS["mid_budget"]          = (PROJECT_INPUTS["min_budget"]  + PROJECT_INPUTS["max_budget"])  / 2
PROJECT_INPUTS["mid_cost_floor_plate"]= (PROJECT_INPUTS["min_cost_floor_plate"] + PROJECT_INPUTS["max_cost_floor_plate"]) / 2
PROJECT_INPUTS["mid_glazing_ratio"]   = (PROJECT_INPUTS["min_glazing_ratio"] + PROJECT_INPUTS["max_glazing_ratio"]) / 2

# ── Print summary ──────────────────────────────────────────────────────────────
print(f"\nProject type : {PROJECT_INPUTS['project_type']}")
print(f"Area range   : {PROJECT_INPUTS['min_area_m2']:,.0f} – {PROJECT_INPUTS['max_area_m2']:,.0f} m²")
print(f"Budget range : £{PROJECT_INPUTS['min_budget']:,.0f} – £{PROJECT_INPUTS['max_budget']:,.0f}")
print(f"Floor cost   : £{PROJECT_INPUTS['min_cost_floor_plate']:,.0f} – £{PROJECT_INPUTS['max_cost_floor_plate']:,.0f} /m²")
print(f"Glazing ratio: {PROJECT_INPUTS['min_glazing_ratio']:.0%} – {PROJECT_INPUTS['max_glazing_ratio']:.0%}")
print(f"Floor height : {PROJECT_INPUTS['min_floor_height_m']}m – {PROJECT_INPUTS['max_floor_height_m']}m")
print(f"Occupancy    : {PROJECT_INPUTS['min_occupancy_per_m2']} – {PROJECT_INPUTS['max_occupancy_per_m2']} occ/m²")

---
## Cell 3 — Document parsers

In [ ]:
import fitz  # PyMuPDF

def parse_excel_to_text(path: Path) -> str:
    """Convert an Excel sheet to a readable text block for agent context."""
    wb     = openpyxl.load_workbook(path, data_only=True)
    ws     = wb.active
    lines  = []
    for row in ws.iter_rows(values_only=True):
        if not any(c is not None for c in row):
            continue
        parts = [str(c).strip() for c in row if c is not None]
        lines.append("  " + "  |  ".join(parts))
    return "\n".join(lines)


# ── Per-agent document text (loaded once, injected into system prompts) ────────
print("Loading agent documents...\n")

agent_docs = {}

for name, path in INPUT_FILES.items():
    if path.exists():
        text = parse_excel_to_text(path)
        agent_docs[name] = {path.stem: text}
        print(f"  ✓ [{name}] {path.name} ({len(text):,} chars)")
    else:
        agent_docs[name] = {}
        print(f"  ✗ [{name}] not found → {path}")

print("\nSummary:")
for agent, docs in agent_docs.items():
    status = ", ".join(docs.keys()) if docs else "no docs"
    print(f"  {agent:<28} → {status}")

---
## Cell 4 — BaseAgent

In [ ]:
class BaseAgent:

    def __init__(self, role, domain_description, docs, bias_instruction):
        self.role               = role
        self.domain_description = domain_description
        self.docs               = docs
        self.bias_instruction   = bias_instruction

    def build_system_prompt(self) -> str:
        parts = [
            f"You are the {self.role} specialist on an architectural design council.",
            f"Your domain: {self.domain_description}",
            self.bias_instruction,
            "",
            "=" * 60,
            PARAMETER_DEFINITIONS,      # ← full definitions injected here
            "=" * 60,
            "",
            "Your council is producing a set of objective weights (summing to 1.0)",
            "for the three parameters defined above:",
            "  1. cost_gbp       — total construction cost in £",
            "  2. carbon_eq      — embodied carbon in kg CO₂e/m³",
            "  3. glazing_ratio  — facade glazing percentage",
            "",
            "These weights feed directly into a Grasshopper multi-objective optimisation algorithm.",
            "Higher weight = that parameter is prioritised by the optimiser.",
            "A high cost_gbp weight → optimiser strongly minimises cost.",
            "A high carbon_eq weight → optimiser strongly minimises embodied carbon.",
            "A high glazing_ratio weight → optimiser strongly targets the optimal glazing range.",
            "",
            "Be direct and evidence-based. Cite specific values from your reference",
            "documents and the project data when making your argument.",
            "Keep responses concise — 150 words max for Stage 1, 100 words max for Stage 2.",
        ]

        if self.docs:
            parts.append("\n── Reference documents ────────────────────────────────")
            for label, text in self.docs.items():
                parts.append(f"\n[{label}]\n{text}")

        return "\n".join(parts)

    def call(self, user_message: str) -> str:
        response = client.chat.completions.create(
            model=MODEL,
            max_tokens=MAX_TOKENS,
            temperature=0.7,
            messages=[
                {"role": "system", "content": self.build_system_prompt()},
                {"role": "user",   "content": user_message},
            ],
        )
        return response.choices[0].message.content.strip()

    def __repr__(self):
        return f"<Agent: {self.role}>"

print("BaseAgent defined.")

---
## Cell 5 — Specialist agents

In [ ]:
# ── Helpers ────────────────────────────────────────────────────────────────────
def _brief_text() -> str:
    """Concatenate all design brief documents into one context block."""
    docs = agent_docs.get("design_brief", {})
    if not docs:
        return "(No brief document uploaded — reasoning from project inputs only.)"
    return "\n\n".join(f"[{k}]\n{v}" for k, v in docs.items())


def _cost_context() -> str:
    p = PROJECT_INPUTS
    return f"""
Project cost context
────────────────────
Project type         : {p['project_type']}
Area range           : {p['min_area_m2']:,.0f} – {p['max_area_m2']:,.0f} m²  (mid: {p['mid_area_m2']:,.0f} m²)
Budget range         : £{p['min_budget']:,.0f} – £{p['max_budget']:,.0f}  (mid: £{p['mid_budget']:,.0f})
Floor plate cost     : £{p['min_cost_floor_plate']:,.0f} – £{p['max_cost_floor_plate']:,.0f} /m²
Glass facade cost    : £{p['cost_glass_facade']:,.0f} /m²  |  Solid facade: £{p['cost_solid_facade']:,.0f} /m²
Material costs       : Concrete £{p['material_cost_concrete']:,.0f}/m³  |  Timber £{p['material_cost_timber']:,.0f}/m³  |  Steel £{p['material_cost_steel']:,.0f}/m³
Glazing ratio range  : {p['min_glazing_ratio']:.0%} – {p['max_glazing_ratio']:.0%}
Embodied carbon      : Concrete {p['concrete_embodied_carbon']} kg CO₂e/m³  |  Timber {p['timber_embodied_carbon']} kg CO₂e/m³  |  Steel {p['steel_embodied_carbon']} kg CO₂e/m³
Floor height range   : {p['min_floor_height_m']}m – {p['max_floor_height_m']}m
Occupancy range      : {p['min_occupancy_per_m2']} – {p['max_occupancy_per_m2']} occupants/m²

Note: all values are ranges — agents should reason about trade-offs across the range,
not assume a single fixed value.
""".strip()


# ── Agent factory ──────────────────────────────────────────────────────────────
def build_agents() -> dict[str, BaseAgent]:

    design_brief_agent = BaseAgent(
        role="Design Brief",
        domain_description=(
            "Client intent, spatial requirements, programme, room adjacency, "
            "user experience, and architectural quality."
        ),
        docs={
            "client_brief": _brief_text(),
        },
        bias_instruction=(
            "You advocate for faithful interpretation of the client brief. "
            "Spatial quality and programme compliance must not be sacrificed for cost or structure."
        ),
    )

    sustainability_agent = BaseAgent(
        role="Sustainability",
        domain_description=(
            "Environmental performance, embodied carbon, operational energy, "
            "BREEAM / LEED certification, and net-zero strategy."
        ),
        docs={
            **agent_docs.get("sustainability", {}),
            "project_target": (
                f"Glazing ratio range: {PROJECT_INPUTS['min_glazing_ratio']:.0%} – {PROJECT_INPUTS['max_glazing_ratio']:.0%}\n"
                f"Concrete embodied carbon: {PROJECT_INPUTS['concrete_embodied_carbon']} kg CO₂e/m³\n"
                f"Timber embodied carbon:   {PROJECT_INPUTS['timber_embodied_carbon']} kg CO₂e/m³\n"
                f"Steel embodied carbon:    {PROJECT_INPUTS['steel_embodied_carbon']} kg CO₂e/m³"
            ),
        },
        bias_instruction=(
            "You push hard for environmental performance. "
            "Embodied carbon and glazing ratio are your primary levers — "
            "argue for material choices and glazing ranges that minimise carbon impact."
        ),
    )

    cost_agent = BaseAgent(
        role="Cost Consultancy",
        domain_description=(
            "Budget management, cost planning, quantity surveying, "
            "value engineering, and whole-life cost."
        ),
        docs={
            # Computed cost summary is always injected
            "cost_breakdown": _cost_context(),
            # Plus any optional QS docs loaded in Cell 3
            **agent_docs.get("cost_consultancy", {}),
        },
        bias_instruction=(
            "You are a pragmatic QS. Budget discipline is your primary duty. "
            "Any weight configuration that risks exceeding the budget envelope should be penalised."
        ),
    )

    return {
        "design_brief":          design_brief_agent,
        "sustainability":         sustainability_agent,
        "cost_consultancy":       cost_agent,
    }


agents = build_agents()
print("Agents built:\n")
for name, agent in agents.items():
    doc_labels = list(agent.docs.keys())
    chars      = sum(len(v) for v in agent.docs.values())
    print(f"  {agent.role:<28} docs: {doc_labels}  ({chars:,} chars injected)")

---
## Cell 6 — chairperson Agent

In [ ]:
CHAIRMAN_SYSTEM_PROMPT = f"""
You are the Chairperson of the Architecture Design Council.
You have received Stage 1 opinions and Stage 2 rebuttals from three specialist agents:
Design Brief, Sustainability, and Cost Consultancy.

{PARAMETER_DEFINITIONS}

Your task is to synthesise the debate and deliver a final verdict on the objective
weights for the Grasshopper multi-objective optimisation algorithm.

Rules:
  - Weights must sum exactly to 1.0
  - Each weight must be between 0.05 and 0.60
  - State your final weights clearly in this exact format:
      FINAL_WEIGHTS: cost_gbp=X.XX, carbon_eq=X.XX, glazing_ratio=X.XX
  - Follow with 3–4 sentences explaining your reasoning
  - Do not output JSON — plain text only
""".strip()


def parse_weights_from_text(text: str) -> dict:
    match = re.search(r"FINAL_WEIGHTS:\s*(.+?)(?:\n|$)", text.strip(), re.IGNORECASE)
    if not match:
        raise ValueError("No FINAL_WEIGHTS line found in chairperson response.")

    weights = {}
    for pair in match.group(1).split(","):
        pair = pair.strip()
        if "=" in pair:
            key, val = pair.split("=", 1)
            weights[key.strip()] = round(float(val.strip()), 4)

    total = sum(weights.values())
    if not (0.98 <= total <= 1.02):
        raise ValueError(f"Weights sum to {total:.3f}, expected 1.0")

    required = {"cost_gbp", "carbon_eq", "glazing_ratio"}
    missing  = required - set(weights.keys())
    if missing:
        raise ValueError(f"Missing parameters: {missing}")

    return weights


def build_chairperson_transcript(stage1: dict, stage2: dict) -> str:
    """
    Build a clean, condensed transcript for the chairperson.
    Extracts only the WEIGHTS/ARGUMENT from Stage 1 and the
    REVISED_WEIGHTS/REASONING from Stage 2 — strips all the
    markdown headers and log noise that bloat the context window.
    """
    lines = ["COUNCIL DEBATE SUMMARY", "=" * 50, ""]

    lines.append("STAGE 1 — INITIAL POSITIONS")
    lines.append("-" * 30)
    for role, text in stage1.items():
        lines.append(f"\n{role.upper()}:")
        # Extract just WEIGHTS and ARGUMENT lines
        for line in text.splitlines():
            if any(line.startswith(tag) for tag in ("WEIGHTS:", "ARGUMENT:")):
                lines.append(f"  {line}")

    lines.append("\n\nSTAGE 2 — AFTER DEBATE")
    lines.append("-" * 30)
    for role, text in stage2.items():
        lines.append(f"\n{role.upper()}:")
        # Extract just the key negotiation lines
        for line in text.splitlines():
            if any(line.startswith(tag) for tag in
                   ("CONCESSION:", "CHALLENGE:", "REVISED_WEIGHTS:", "REASONING:")):
                lines.append(f"  {line}")

    return "\n".join(lines)


def call_chairperson(stage1: dict, stage2: dict, human_instruction: str = "") -> dict:
    condensed = build_chairperson_transcript(stage1, stage2)

    # Add human instruction block if present
    human_block = ""
    if human_instruction:
        human_block = (
            f"\nSTAKEHOLDER INSTRUCTION (must be reflected in final weights)\n"
            f"{'─' * 50}\n"
            f"{human_instruction}\n"
            f"{'─' * 50}\n"
        )

    user_msg = (
        f"{condensed}\n"
        f"{human_block}\n"
        f"Project context:\n{_cost_context()}\n\n"
        "Based on the debate above and any stakeholder instruction, deliver your final verdict.\n"
        "Start your response with exactly:\n"
        "FINAL_WEIGHTS: cost_gbp=X.XX, carbon_eq=X.XX, glazing_ratio=X.XX\n"
        "Then explain your reasoning in 3–4 sentences, specifically referencing "
        "the stakeholder instruction if one was provided."
    )

    for attempt in range(2):
        response = client.chat.completions.create(
            model=MODEL,
            max_tokens=MAX_TOKENS,
            temperature=0.3,
            messages=[
                {"role": "system", "content": CHAIRMAN_SYSTEM_PROMPT},
                {"role": "user",   "content": user_msg},
            ],
        )
        raw = response.choices[0].message.content.strip()

        print(f"\n── CHAIRPERSON RAW OUTPUT (attempt {attempt+1}) ──────────")
        print(raw)
        print("─────────────────────────────────────────────────────────\n")

        try:
            weights = parse_weights_from_text(raw)
            return {"weights": weights, "reasoning": raw}
        except (ValueError, KeyError) as e:
            if attempt == 0:
                print(f"  ⚠ Parse failed ({e}), retrying…")
                user_msg += (
                    "\n\nYour response could not be parsed. "
                    "You MUST start with this exact line:\n"
                    "FINAL_WEIGHTS: cost_gbp=X.XX, carbon_eq=X.XX, glazing_ratio=X.XX"
                )
            else:
                raise RuntimeError(
                    f"Chairperson failed after 2 attempts.\nRaw output:\n{raw}"
                ) from e

print("Chairperson defined.")

---
## Cell 7 — Council orchestrator

In [ ]:
from IPython.display import display, Markdown
from datetime import datetime, timezone

def build_stage1_prompt() -> str:
    p = PROJECT_INPUTS
    return f"""
PROJECT TYPE : {p['project_type']}
AREA RANGE   : {p['min_area_m2']:,.0f} – {p['max_area_m2']:,.0f} m²
BUDGET RANGE : £{p['min_budget']:,.0f} – £{p['max_budget']:,.0f}
FLOOR COST   : £{p['min_cost_floor_plate']:,.0f} – £{p['max_cost_floor_plate']:,.0f} /m²
GLAZING RATIO: {p['min_glazing_ratio']:.0%} – {p['max_glazing_ratio']:.0%}
FLOOR HEIGHT : {p['min_floor_height_m']}m – {p['max_floor_height_m']}m
OCCUPANCY    : {p['min_occupancy_per_m2']} – {p['max_occupancy_per_m2']} occ/m²
MATERIALS    : Concrete £{p['material_cost_concrete']:,.0f}/m³  |  {p['concrete_embodied_carbon']} kg CO₂e/m³
               Timber   £{p['material_cost_timber']:,.0f}/m³   |  {p['timber_embodied_carbon']} kg CO₂e/m³
               Steel    £{p['material_cost_steel']:,.0f}/m³    |  {p['steel_embodied_carbon']} kg CO₂e/m³
FACADE COSTS : Glass £{p['cost_glass_facade']:,.0f}/m²  |  Solid £{p['cost_solid_facade']:,.0f}/m²

STAGE 1 TASK
────────────
Review your reference documents and the project context above.
All values are ranges — reason about trade-offs across the full range.

Propose weights for the three parameters below, summing to 1.0:
  cost_gbp      — how much to prioritise minimising total construction cost (£)
  carbon_eq     — how much to prioritise minimising embodied carbon (kg CO₂e/m³)
  glazing_ratio — how much to prioritise optimising the facade glazing percentage

Respond in this format ONLY — no extra commentary:
WEIGHTS: cost_gbp=X.XX, carbon_eq=X.XX, glazing_ratio=X.XX
ARGUMENT: <your 100–150 word case citing specific £ and kg CO₂e values from above>
""".strip()


def build_stage2_prompt(stage1_responses: dict, current_role: str) -> str:
    
    own_response = stage1_responses.get(current_role, "")
    
    other_responses = "\n\n".join(
        f"── {role.upper()} AGENT ──\n{text}"
        for role, text in stage1_responses.items()
        if role != current_role          # ← exclude own response from "others" block
    )

    return f"""
STAGE 2 — DEBATE & COMPROMISE
──────────────────────────────
YOUR Stage 1 position was:
{own_response}

The OTHER council members proposed the following in Stage 1:
{other_responses}

You are now entering a structured negotiation. Genuinely engage with the other
agents' arguments — do not simply restate your Stage 1 position.
...

Respond to ALL FOUR of the following:

1. CONCESSION
   Name one specific value or threshold from your domain (a £/m² figure, a kg CO₂e
   limit, or a glazing % boundary) that you are willing to relax, and explain why.

2. CHALLENGE
   Name one specific claim from another agent — cite their exact figure — and explain
   why it is overstated or unrealistic given the project ranges.

3. REVISED_WEIGHTS
   Update your weights to reflect your concession and challenge. Must sum to 1.0.
   If unchanged, explain precisely why no movement is possible.

4. REASONING
   2–3 sentences connecting your concession and challenge to your revised weights.

Keep each section to 2–3 sentences. Complete all four sections in one response.

Respond in this format ONLY:
CONCESSION: <specific parameter/value you are softening and why>
CHALLENGE: <agent name, their specific figure, why it is overstated>

    REVISED_WEIGHTS: cost_gbp=X.XX, carbon_eq=X.XX, glazing_ratio=X.XX
    REASONING: <how your concession and challenge justify the revised weights>
""".strip()


STAGE2_REQUIRED = ["CONCESSION:", "CHALLENGE:", "REVISED_WEIGHTS:", "REASONING:"]


def run_council(agents: dict, verbose: bool = True) -> dict:
    transcript_lines = []

    def log(text):
        transcript_lines.append(text)
        if verbose:
            display(Markdown(text))

    # ── Stage 1 ────────────────────────────────────────────────────────────────
    log("## Stage 1 — First opinions\n")
    stage1_prompt    = build_stage1_prompt()
    stage1_responses = {}

    for role, agent in agents.items():
        log(f"### {agent.role}")
        print(f"  Calling {agent.role}…", end=" ", flush=True)
        try:
            response = agent.call(stage1_prompt)
            stage1_responses[role] = response
            log(response + "\n")
            print("✓")
        except Exception as e:
            print(f"✗ FAILED: {e}")
            stage1_responses[role] = f"[Agent failed to respond: {e}]"
            log(f"*{agent.role} failed: {e}*\n")

    # ── Stage 2 ────────────────────────────────────────────────────────────────
    log("---\n## Stage 2 — Debate (1 round)\n")
    stage2_prompt    = build_stage2_prompt(stage1_responses)
    stage2_responses = {}

    for role, agent in agents.items():
        log(f"### {agent.role}")
        print(f"  Calling {agent.role}…", end=" ", flush=True)

        response   = ""
        last_error = ""

        for attempt in range(3):
            try:
                candidate = agent.call(stage2_prompt)
                missing   = [tag for tag in STAGE2_REQUIRED if tag not in candidate]

                if missing:
                    last_error = f"missing sections: {missing}"
                    if attempt < 2:
                        print(f"\n    ⚠ Attempt {attempt+1} incomplete ({last_error}), retrying…", end=" ")
                        continue

                if "WEIGHTS_UNCHANGED" not in candidate:
                    rw_match = re.search(r"REVISED_WEIGHTS:\s*(.+)", candidate, re.IGNORECASE)
                    if not rw_match:
                        last_error = "REVISED_WEIGHTS line not parseable"
                        if attempt < 2:
                            print(f"\n    ⚠ Attempt {attempt+1}: {last_error}, retrying…", end=" ")
                            continue

                response = candidate
                break

            except Exception as e:
                last_error = str(e)
                if attempt < 2:
                    print(f"\n    ⚠ Attempt {attempt+1} errored ({e}), retrying…", end=" ")

        if not response:
            response = f"[{agent.role} failed after 3 attempts — last error: {last_error}]"
            print(f"✗ FAILED ({last_error})")
        else:
            print("✓")

        stage2_responses[role] = response
        log(response + "\n")

    # ── Chairperson ────────────────────────────────────────────────────────────
    log("---\n## Chairperson — Final synthesis\n")
    full_transcript = "\n\n".join(transcript_lines)

    print("  Calling Chairperson…", end=" ", flush=True)
    try:
        result = call_chairman(full_transcript)
        print("✓")
    except RuntimeError as e:
        print("✗ FAILED")
        raise e

    result["metadata"] = {
        "project":       PROJECT_INPUTS["project_type"],
        "generated_at":  datetime.now(timezone.utc).isoformat(),
        "model":         MODEL,
        "lm_studio_url": LM_STUDIO_BASE_URL,
        "debate_rounds": 1,
        "agents":        list(agents.keys()),
    }

    result["_transcript"] = full_transcript
    return result


print("Orchestrator defined. Run Cell 8 to start the council.")

---
## Cell 8a — Council Run — Stage 1 — Initial Opinions

This will make **9 API calls** total (4 Stage 1 + 4 Stage 2 + 1 chairperson).  
Each response streams into the cell output as it arrives.

In [ ]:
# ── Stage 1 — Run independently ───────────────────────────────────────────────
# human_input_expanded is set by Cell 8b.5 — empty on first run
if "human_input_expanded" not in dir():
    human_input_expanded = ""
if "human_input_raw" not in dir():
    human_input_raw      = ""
if "rerun_requested" not in dir():
    rerun_requested      = False

transcript_lines = []

def log(text):
    transcript_lines.append(text)
    display(Markdown(text))


def build_stage1_prompt() -> str:
    p = PROJECT_INPUTS

    human_block = ""
    if human_input_expanded:
        human_block = f"""
STAKEHOLDER INSTRUCTION (must be taken into account)
─────────────────────────────────────────────────────
{human_input_expanded}
─────────────────────────────────────────────────────
"""

    return f"""
PROJECT TYPE : {p['project_type']}
AREA RANGE   : {p['min_area_m2']:,.0f} – {p['max_area_m2']:,.0f} m²
BUDGET RANGE : £{p['min_budget']:,.0f} – £{p['max_budget']:,.0f}
FLOOR COST   : £{p['min_cost_floor_plate']:,.0f} – £{p['max_cost_floor_plate']:,.0f} /m²
GLAZING RATIO: {p['min_glazing_ratio']:.0%} – {p['max_glazing_ratio']:.0%}
FLOOR HEIGHT : {p['min_floor_height_m']}m – {p['max_floor_height_m']}m
OCCUPANCY    : {p['min_occupancy_per_m2']} – {p['max_occupancy_per_m2']} occ/m²
MATERIALS    : Concrete £{p['material_cost_concrete']:,.0f}/m³  |  {p['concrete_embodied_carbon']} kg CO₂e/m³
               Timber   £{p['material_cost_timber']:,.0f}/m³   |  {p['timber_embodied_carbon']} kg CO₂e/m³
               Steel    £{p['material_cost_steel']:,.0f}/m³    |  {p['steel_embodied_carbon']} kg CO₂e/m³
FACADE COSTS : Glass £{p['cost_glass_facade']:,.0f}/m²  |  Solid £{p['cost_solid_facade']:,.0f}/m²
{human_block}
STAGE 1 TASK
────────────
Review your reference documents and the project context above.
All values are ranges — reason about trade-offs across the full range.

Propose weights for the three parameters below, summing to 1.0:
  cost_gbp      — how much to prioritise minimising total construction cost (£)
  carbon_eq     — how much to prioritise minimising embodied carbon (kg CO₂e/m³)
  glazing_ratio — how much to prioritise optimising the facade glazing percentage

Respond in this format ONLY — no extra commentary:
WEIGHTS: cost_gbp=X.XX, carbon_eq=X.XX, glazing_ratio=X.XX
ARGUMENT: <your 100–150 word case citing specific £ and kg CO₂e values from above>
""".strip()


log("## Stage 1 — First opinions\n")
stage1_prompt    = build_stage1_prompt()
stage1_responses = {}

for role, agent in agents.items():
    log(f"### {agent.role}")
    print(f"  Calling {agent.role}…", end=" ", flush=True)
    try:
        response = agent.call(stage1_prompt)
        stage1_responses[role] = response
        log(response + "\n")
        print("✓")
    except Exception as e:
        print(f"✗ FAILED: {e}")
        stage1_responses[role] = f"[Agent failed: {e}]"
        log(f"*{agent.role} failed: {e}*\n")

print("\nStage 1 complete. Run Cell 8b for Stage 2.")

---
## Cell 8b — Council Run — Stage 2 — Debate

This will make **9 API calls** total (4 Stage 1 + 4 Stage 2 + 1 chairperson).  
Each response streams into the cell output as it arrives.

In [ ]:
# ── Stage 2 — Run independently (requires Cell 8a) ────────────────────────────
assert stage1_responses, "Run Cell 8a first."


def build_stage2_prompt(stage1_responses: dict, current_role: str) -> str:

    own_response = stage1_responses.get(current_role, "")

    other_responses = "\n\n".join(
        f"── {role.upper()} AGENT ──\n{text}"
        for role, text in stage1_responses.items()
        if role != current_role
    )

    return f"""

STAGE 2 — DEBATE & COMPROMISE
──────────────────────────────
YOUR Stage 1 position was:
{own_response}

The OTHER council members proposed the following in Stage 1:
{other_responses}

You are now entering a structured negotiation. Genuinely engage with the other
agents' arguments — do not simply restate your Stage 1 position.

Respond to ALL FOUR of the following:

1. CONCESSION
   Name one specific value or threshold from your domain (a £/m² figure, a kg CO₂e
   limit, or a glazing % boundary) that you are willing to relax, and explain why.

2. CHALLENGE
   Name one specific claim from another agent — cite their exact figure — and explain
   why it is overstated or unrealistic given the project ranges.

3. REVISED_WEIGHTS
   Update your weights to reflect your concession and challenge. Must sum to 1.0.
   If unchanged, explain precisely why no movement is possible.

4. REASONING
   2–3 sentences connecting your concession and challenge to your revised weights.

Keep each section to 2–3 sentences. Complete all four sections in one response.

Respond in this format ONLY:
CONCESSION: <specific parameter/value you are softening and why>
CHALLENGE: <agent name, their specific figure, why it is overstated>
REVISED_WEIGHTS: cost_gbp=X.XX, carbon_eq=X.XX, glazing_ratio=X.XX
REASONING: <how your concession and challenge justify the revised weights>
""".strip()


log("---\n## Stage 2 — Debate (1 round)\n")

# Initialise here so it always exists even if agents fail
stage2_responses = {}

STAGE2_REQUIRED = ["CONCESSION:", "CHALLENGE:", "REVISED_WEIGHTS:", "REASONING:"]

for role, agent in agents.items():
    log(f"### {agent.role}")
    print(f"  Calling {agent.role}…", end=" ", flush=True)

    stage2_prompt = build_stage2_prompt(stage1_responses, current_role=role)

    response   = ""
    last_error = ""

    for attempt in range(3):
        try:
            candidate = agent.call(stage2_prompt)
            missing   = [tag for tag in STAGE2_REQUIRED if tag not in candidate]

            if missing:
                last_error = f"missing sections: {missing}"
                if attempt < 2:
                    print(f"\n    ⚠ Attempt {attempt+1} incomplete ({last_error}), retrying…", end=" ")
                    continue

            if "WEIGHTS_UNCHANGED" not in candidate:
                rw_match = re.search(r"REVISED_WEIGHTS:\s*(.+)", candidate, re.IGNORECASE)
                if not rw_match:
                    last_error = "REVISED_WEIGHTS line not parseable"
                    if attempt < 2:
                        print(f"\n    ⚠ Attempt {attempt+1}: {last_error}, retrying…", end=" ")
                        continue

            response = candidate
            break

        except Exception as e:
            last_error = str(e)
            if attempt < 2:
                print(f"\n    ⚠ Attempt {attempt+1} errored ({e}), retrying…", end=" ")

    if not response:
        response = f"[{agent.role} failed after 3 attempts — last error: {last_error}]"
        print(f"✗ FAILED ({last_error})")
    else:
        print("✓")

    stage2_responses[role] = response
    log(response + "\n")
    transcript_lines.append(f"### {agent.role}\n{response}\n")

print("\nStage 2 complete. Run Cell 8b.5 for human input.")

---
## Cell 8b.5 — User Input — Stage 2

In [ ]:
# ── Cell 8b.5 — Human input to Chairperson ────────────────────────────────────
assert stage1_responses, "Run Cell 8a first."
assert stage2_responses, "Run Cell 8b first."


def expand_human_input(raw: str) -> str:
    """
    Expand a short user comment into a clear, professionally
    phrased 1–2 sentence instruction for the agents.
    """
    if not raw.strip():
        return ""

    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=200,
        temperature=0.4,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a professional architectural project manager. "
                    "A user has provided a brief comment or instruction regarding "
                    "an architectural design council debate. Expand their input into "
                    "a clear, professionally phrased instruction of maximum 2 sentences "
                    "that can be understood by specialist agents (cost, sustainability, "
                    "design brief). Do not add opinions — only clarify and expand what "
                    "the user said. Output only the expanded instruction, nothing else."
                ),
            },
            {
                "role": "user",
                "content": f"User comment: {raw.strip()}"
            },
        ],
    )
    return response.choices[0].message.content.strip()


def ask_chairperson_for_input(stage1: dict, stage2: dict) -> str:
    """
    Ask the chairperson to formulate one focused question for the
    human stakeholder based on the debate so far.
    """
    condensed = build_chairperson_transcript(stage1, stage2)

    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=300,
        temperature=0.4,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are the Chairperson of an architectural design council. "
                    "You have reviewed a debate between specialist agents on cost, "
                    "carbon, and glazing weights for a building optimisation. "
                    "Before delivering your final verdict, you want to ask the human "
                    "stakeholder one specific, focused question about their priorities "
                    "or preferences that would help you make a better final decision. "
                    "Ask only ONE question. Keep it concise and specific to the debate. "
                    "Output only the question, nothing else."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"Here is the debate summary:\n\n{condensed}\n\n"
                    "What one question would you ask the human stakeholder?"
                ),
            },
        ],
    )
    return response.choices[0].message.content.strip()


# ── Step 1: Chairperson asks a focused question ────────────────────────────────
print("=" * 60)
print("CHAIRPERSON — Human input requested")
print("=" * 60)

chairperson_question = ask_chairperson_for_input(stage1_responses, stage2_responses)
print(f"\nChairperson asks:\n\n  → {chairperson_question}\n")

# ── Step 2: Collect human input ────────────────────────────────────────────────
print("Enter your response below (or press Enter to skip):")
human_input_raw = input("Your input: ").strip()

if not human_input_raw:
    print("\nNo input provided — Chairperson will proceed independently.")
    human_input_expanded = ""
    rerun_requested      = False
else:
    # Expand the raw input into professional language
    print("\nExpanding your input…", end=" ", flush=True)
    human_input_expanded = expand_human_input(human_input_raw)
    print("✓")
    print(f"\nExpanded instruction:\n  → {human_input_expanded}\n")

    # ── Step 3: Ask if user wants to rerun the council ─────────────────────────
    print("Would you like to rerun the council debate with this input included?")
    print("  [y] Yes — rerun Stages 1 & 2 with your input given to all agents")
    print("  [n] No  — Chairperson will factor it in directly for the final verdict")
    rerun_choice    = input("\nYour choice (y/n): ").strip().lower()
    rerun_requested = rerun_choice == "y"

    if rerun_requested:
        print("\n✓ Rerun requested.")
        print("  → Re-run Cell 8a then Cell 8b — your input will be included.")
        print("  → Then run Cell 8b.5 again, then Cell 8c for the final verdict.")
    else:
        print("\n✓ No rerun — Chairperson will factor your input into the final verdict.")

print("\n" + "=" * 60)
print(f"Human input raw     : {human_input_raw or '(none)'}")
print(f"Human input expanded: {human_input_expanded or '(none)'}")
print(f"Rerun requested     : {rerun_requested}")
print("=" * 60)
print("\nRun Cell 8c for the final verdict.")

---
## Cell 8c — Council Run — Stage 3 — Chairperson Verdict

This will make **9 API calls** total (4 Stage 1 + 4 Stage 2 + 1 chairperson).  
Each response streams into the cell output as it arrives.

In [ ]:
# ── DEBUG — run this before Cell 8c to see raw chairperson output ─────────────
full_transcript = "\n\n".join(transcript_lines)

user_msg = (
    f"Here is the full debate transcript:\n\n{full_transcript}\n\n"
    f"Project context:\n{_cost_context()}\n\n"
    "Now deliver your final verdict. Start with the FINAL_WEIGHTS line, then your reasoning."
)

raw_response = client.chat.completions.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    temperature=0.3,
    messages=[
        {"role": "system", "content": CHAIRMAN_SYSTEM_PROMPT},
        {"role": "user",   "content": user_msg},
    ],
)

raw = raw_response.choices[0].message.content.strip()
print("── RAW CHAIRPERSON OUTPUT ──────────────────────────────")
print(raw)
print("────────────────────────────────────────────────────────")
print(f"\nCharacters: {len(raw)}")
print(f"Contains FINAL_WEIGHTS: {'FINAL_WEIGHTS' in raw}")

In [ ]:
# ── Chairperson — Run independently (requires 8a + 8b + 8b.5) ────────────────
assert stage1_responses, "Run Cell 8a first."
assert stage2_responses, "Run Cell 8b first."

log("---\n## Chairperson — Final synthesis\n")

# Surface the human input in the transcript if present
if human_input_expanded:
    log(f"**Stakeholder instruction passed to Chairperson:**\n\n> {human_input_expanded}\n")
else:
    log("*No stakeholder instruction provided — Chairperson proceeding independently.*\n")

print("  Calling Chairperson…", end=" ", flush=True)
try:
    result = call_chairperson(
        stage1_responses,
        stage2_responses,
        human_instruction=human_input_expanded,
    )
    print("✓")
except RuntimeError as e:
    print("✗ FAILED")
    raise e

# ── Display chairperson verdict inline ────────────────────────────────────────
weights   = result["weights"]
reasoning = result["reasoning"]

log("### Chairperson verdict\n")
log(
    "| Parameter | Weight |\n| --- | --- |\n" +
    "\n".join(
        f"| **{k}** | `{v:.2f}` |"
        for k, v in sorted(weights.items(), key=lambda x: -x[1])
    )
)
log(f"\n**Reasoning:**\n\n{reasoning}\n")

# ── Attach metadata ────────────────────────────────────────────────────────────
result["metadata"] = {
    "project":               PROJECT_INPUTS["project_type"],
    "generated_at":          datetime.now(timezone.utc).isoformat(),
    "model":                 MODEL,
    "lm_studio_url":         LM_STUDIO_BASE_URL,
    "debate_rounds":         1,
    "agents":                list(agents.keys()),
    "human_input_raw":       human_input_raw,
    "human_input_expanded":  human_input_expanded,
    "rerun_requested":       rerun_requested,
}

result["_transcript"] = build_chairperson_transcript(stage1_responses, stage2_responses)

print("\nChairperson complete. Run Cell 9 to write weights_output.json.")
print("Human input will be flushed after Cell 9 confirms both files are written.")

---
## Cell 9 — Inspect results + write weights_output.json

In [ ]:
weights   = result["weights"]
reasoning = result["reasoning"]

output_data = {
    "objective_weights": {
        "cost_gbp":      weights.get("cost_gbp",      0.0),
        "carbon_eq":     weights.get("carbon_eq",     0.0),
        "glazing_ratio": weights.get("glazing_ratio",  0.0),
    },
    "parameter_definitions": {
        "cost_gbp":      "Total construction cost in £ (floor plate + facade)",
        "carbon_eq":     "Embodied carbon in kg CO₂e/m³ across all materials",
        "glazing_ratio": "Facade glazing percentage (transparent vs opaque area)",
    },
    "project_ranges": {
        "budget_range_gbp":       [PROJECT_INPUTS["min_budget"],          PROJECT_INPUTS["max_budget"]],
        "floor_cost_range_gbp_m2":[PROJECT_INPUTS["min_cost_floor_plate"],PROJECT_INPUTS["max_cost_floor_plate"]],
        "glazing_ratio_range":    [PROJECT_INPUTS["min_glazing_ratio"],   PROJECT_INPUTS["max_glazing_ratio"]],
        "carbon_per_material": {
            "concrete": PROJECT_INPUTS["concrete_embodied_carbon"],
            "timber":   PROJECT_INPUTS["timber_embodied_carbon"],
            "steel":    PROJECT_INPUTS["steel_embodied_carbon"],
        },
    },
    "reasoning":  reasoning,
    "metadata":   result["metadata"],
}

display(Markdown("## Final weights"))
display(Markdown(
    "| Parameter | Weight | Definition |\n| --- | --- | --- |\n" +
    "\n".join(
        f"| **{k}** | `{v:.2f}` | {output_data['parameter_definitions'][k]} |"
        for k, v in sorted(weights.items(), key=lambda x: -x[1])
    )
))
display(Markdown(f"**Chairperson reasoning:**\n\n{reasoning}"))

# ── at the very end of Cell 9, after both file writes ─────────────────────────

# Write JSON
with open(OUTPUT_PATH, "w") as f:
    json.dump(output_data, f, indent=2)
print(f"✓ JSON written to : {OUTPUT_PATH}")

# Write TXT
txt_path = OUTPUT_PATH.with_suffix(".txt")
with open(txt_path, "w") as f:
    f.write(f"cost_gbp={weights['cost_gbp']:.2f}\n")
    f.write(f"carbon_eq={weights['carbon_eq']:.2f}\n")
    f.write(f"glazing_ratio={weights['glazing_ratio']:.2f}\n")
print(f"✓ TXT written to  : {txt_path}")

# ── Flush human input ONLY after both files are confirmed written ──────────────
human_input_raw      = ""
human_input_expanded = ""
rerun_requested      = False
print("\n✓ Human input flushed — ready for next council run.")

---
## Cell 10 — Optimisation validation + chairperson review

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "ipywidgets"])

# ── Cell 10 — Validate optimisation results against agent documents ────────────
import ipywidgets as widgets
from IPython.display import display, Markdown

# ── Step 1: Load optimisation results ─────────────────────────────────────────
OPT_RESULTS_PATH = Path(r"/Output/Optimization Results.xlsx")

def load_optimisation_results(path: Path) -> dict:
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb.active
    results = {}
    for row in ws.iter_rows(values_only=True):
        if not any(c is not None for c in row):
            continue
        key = str(row[0]).strip() if row[0] is not None else None
        val = row[1] if len(row) > 1 else None
        unit = row[2] if len(row) > 2 else None
        if key is None:
            continue
        try:
            results[key] = float(val)
        except (TypeError, ValueError):
            results[key] = str(val).strip() if val is not None else ""
        if unit is not None:
            results[f"{key}__unit"] = str(unit).strip()
    return results

opt = load_optimisation_results(OPT_RESULTS_PATH)

display(Markdown("## Optimisation results received from Grasshopper"))
display(Markdown(
    "| Parameter | Value | Unit |\n| --- | --- | --- |\n" +
    "\n".join(
        f"| `{k}` | **{v}** | {opt.get(k+'__unit', '—')} |"
        for k, v in opt.items() if not k.endswith("__unit")
    )
))

# ── Step 2: Validate against document ranges ───────────────────────────────────
p = PROJECT_INPUTS

VALIDATION_RULES = [
    {
        "parameter":   "glazing_ratio",
        "value":        opt.get("glazing_ratio"),
        "min":          p["min_glazing_ratio"],
        "max":          p["max_glazing_ratio"],
        "unit":         "%",
        "source":       "Sustainability.xlsx",
        "description":  "Glazing ratio must be within permitted range for BREEAM daylighting compliance.",
        "format":       lambda v: f"{v:.0%}",
        "fmt_min":      lambda v: f"{v:.0%}",
        "fmt_max":      lambda v: f"{v:.0%}",
    },
    {
        "parameter":   "floor height",
        "value":        opt.get("floor height"),
        "min":          p["min_floor_height_m"],
        "max":          p["max_floor_height_m"],
        "unit":         "m",
        "source":       "Building_Regulations.xlsx",
        "description":  "Floor height must comply with building regulations.",
        "format":       lambda v: f"{v}m",
        "fmt_min":      lambda v: f"{v}m",
        "fmt_max":      lambda v: f"{v}m",
    },
    {
        "parameter":   "occupancy",
        "value":        opt.get("occupancy"),
        "min":          p["min_occupancy_per_m2"],
        "max":          p["max_occupancy_per_m2"],
        "unit":         "occ/m²",
        "source":       "Building_Regulations.xlsx",
        "description":  "Occupancy density must comply with building regulations.",
        "format":       lambda v: f"{v} occ/m²",
        "fmt_min":      lambda v: f"{v} occ/m²",
        "fmt_max":      lambda v: f"{v} occ/m²",
    },
    {
        "parameter":   "area",
        "value":        opt.get("area"),
        "min":          p["min_area_m2"],
        "max":          p["max_area_m2"],
        "unit":         "m²",
        "source":       "Client_Brief.xlsx",
        "description":  "Total area must satisfy the client brief GFA requirements.",
        "format":       lambda v: f"{v:,.0f} m²",
        "fmt_min":      lambda v: f"{v:,.0f} m²",
        "fmt_max":      lambda v: f"{v:,.0f} m²",
    },
    {
        "parameter":   "budget",
        "value":        opt.get("budget"),
        "min":          p["min_budget"],
        "max":          p["max_budget"],
        "unit":         "£",
        "source":       "Client_Brief.xlsx",
        "description":  "Project budget must remain within the client approved range.",
        "format":       lambda v: f"£{v:,.0f}",
        "fmt_min":      lambda v: f"£{v:,.0f}",
        "fmt_max":      lambda v: f"£{v:,.0f}",
    },
]

passed = []
failed = []

for rule in VALIDATION_RULES:
    val = rule["value"]
    if val is None:
        failed.append({**rule, "reason": "Parameter not found in optimisation results"})
        continue
    if rule["min"] <= val <= rule["max"]:
        passed.append(rule)
    else:
        direction = "below minimum" if val < rule["min"] else "above maximum"
        rule["reason"] = (
            f"Value {rule['format'](val)} is {direction} "
            f"(allowed: {rule['fmt_min'](rule['min'])} – {rule['fmt_max'](rule['max'])})"
        )
        failed.append(rule)

# ── Step 3: Display validation report ─────────────────────────────────────────
display(Markdown("## Validation report"))

if passed:
    display(Markdown(
        "### ✅ Passed\n"
        "| Parameter | Value | Allowed range | Source |\n| --- | --- | --- | --- |\n" +
        "\n".join(
            f"| `{r['parameter']}` | {r['format'](r['value'])} "
            f"| {r['fmt_min'](r['min'])} – {r['fmt_max'](r['max'])} "
            f"| {r['source']} |"
            for r in passed
        )
    ))

if failed:
    display(Markdown(
        "### ❌ Failed\n"
        "| Parameter | Value | Allowed range | Reason | Source |\n| --- | --- | --- | --- | --- |\n" +
        "\n".join(
            f"| `{r['parameter']}` | {r['format'](r['value']) if r['value'] else '—'} "
            f"| {r['fmt_min'](r['min'])} – {r['fmt_max'](r['max'])} "
            f"| {r['reason']} "
            f"| {r['source']} |"
            for r in failed
        )
    ))
else:
    display(Markdown("### ✅ All parameters passed validation."))

# ── Step 4: Ask chairperson to summarise validation ───────────────────────────
def build_validation_summary_for_chairperson() -> str:
    passed_lines = "\n".join(
        f"  ✓ {r['parameter']}: {r['format'](r['value'])} — within "
        f"{r['fmt_min'](r['min'])} – {r['fmt_max'](r['max'])} ({r['source']})"
        for r in passed
    )
    failed_lines = "\n".join(
        f"  ✗ {r['parameter']}: {r['reason']} ({r['source']}) — {r['description']}"
        for r in failed
    )
    return f"""
OPTIMISATION RESULTS VALIDATION
════════════════════════════════
PASSED ({len(passed)}/{len(VALIDATION_RULES)}):
{passed_lines if passed_lines else '  (none)'}

FAILED ({len(failed)}/{len(VALIDATION_RULES)}):
{failed_lines if failed_lines else '  (none)'}

WEIGHTS USED:
  cost_gbp={result['weights']['cost_gbp']:.2f}
  carbon_eq={result['weights']['carbon_eq']:.2f}
  glazing_ratio={result['weights']['glazing_ratio']:.2f}
""".strip()


validation_summary = build_validation_summary_for_chairperson()

chairperson_validation_response = client.chat.completions.create(
    model=MODEL,
    max_tokens=MAX_TOKENS,
    temperature=0.3,
    messages=[
        {
            "role": "system",
            "content": (
                f"You are the Chairperson of an architectural design council.\n"
                f"{PARAMETER_DEFINITIONS}\n\n"
                "You are reviewing the results of a Grasshopper optimisation run "
                "against the original project documents. Provide a concise assessment "
                "of what passed and what failed, why each failure matters, and what "
                "it implies for the design weights. Keep to 4–6 sentences."
            ),
        },
        {
            "role": "user",
            "content": (
                f"{validation_summary}\n\n"
                f"Project context:\n{_cost_context()}\n\n"
                "Summarise the validation outcome and its implications for the design weights."
            ),
        },
    ],
)

chairperson_validation_text = chairperson_validation_response.choices[0].message.content.strip()
display(Markdown(f"## Chairperson validation assessment\n\n{chairperson_validation_text}"))

# ── Step 5: If failures exist — ask user whether to rerun ─────────────────────
if not failed:
    display(Markdown(
        "## ✅ All parameters validated — no rerun required.\n\n"
        "Run Cell 11 for the final project summary."
    ))
    validation_rerun_requested = False
else:
    display(Markdown(f"### ⚠ {len(failed)} parameter(s) failed — user decision required."))

    rerun_btn     = widgets.Button(description="Yes — Rerun council",  button_style="warning")
    no_rerun_btn  = widgets.Button(description="No — Final summary",   button_style="success")
    decision_area = widgets.Output()

    display(widgets.HBox([rerun_btn, no_rerun_btn]), decision_area)

    def on_rerun(btn):
        global validation_rerun_requested, human_input_expanded
        validation_rerun_requested = True

        # Build a concise validation context to inject into agent prompts
        human_input_expanded = (
            f"A previous optimisation run produced results that failed validation. "
            f"The following issues were identified and must be addressed in your arguments: "
            + " | ".join(f"{r['parameter']}: {r['reason']}" for r in failed)
            + f" Passed checks: "
            + ", ".join(r['parameter'] for r in passed)
            + ". Adjust your weight proposals to help the optimiser avoid these violations."
        )

        with decision_area:
            decision_area.clear_output()
            display(Markdown(
                "**✓ Rerun requested.**\n\n"
                "The validation summary has been added to agent context.\n\n"
                f"> {human_input_expanded}\n\n"
                "Re-run **Cell 8a → 8b → 8b.5 → 8c → 9**, then come back to **Cell 10**."
            ))

    def on_no_rerun(btn):
        global validation_rerun_requested
        validation_rerun_requested = False
        with decision_area:
            decision_area.clear_output()
            display(Markdown(
                "**✓ No rerun.** Run **Cell 11** for the final project summary."
            ))

    rerun_btn.on_click(on_rerun)
    no_rerun_btn.on_click(on_no_rerun)

---
## Cell 11 — Final project summary

In [ ]:
# ── Cell 11 — Final project summary ───────────────────────────────────────────
display(Markdown("# Final project summary"))

# Build a compact context block instead of a long formatted prompt
compact_context = f"""
Project    : {PROJECT_INPUTS['project_type']}
Area       : {PROJECT_INPUTS['min_area_m2']:,.0f}–{PROJECT_INPUTS['max_area_m2']:,.0f} m²
Budget     : £{PROJECT_INPUTS['min_budget']:,.0f}–£{PROJECT_INPUTS['max_budget']:,.0f}
Weights    : cost_gbp={result['weights']['cost_gbp']:.2f}, carbon_eq={result['weights']['carbon_eq']:.2f}, glazing_ratio={result['weights']['glazing_ratio']:.2f}
Opt results: {', '.join(f"{k}={v} {opt.get(k+'__unit','')}" for k, v in opt.items() if not k.endswith('__unit'))}
Passed     : {', '.join(r['parameter'] for r in passed) or 'none'}
Failed     : {', '.join(r['parameter'] for r in failed) or 'none'}
""".strip()

summary_response = client.chat.completions.create(
    model=MODEL,
    max_tokens=350,       # was MAX_TOKENS (1024+) — summary needs far less
    temperature=0.3,
    messages=[
        {
            "role": "system",
            "content": (
                "You are a professional architectural project manager. "
                "Write a concise project summary in exactly 5 bullet points: "
                "1) Project overview  2) Weight rationale  3) Optimisation outcome  "
                "4) Validation status  5) Next steps. "
                "One sentence per bullet. Be specific — cite actual numbers."
            ),
        },
        {
            "role": "user",
            "content": compact_context,
        },
    ],
)

display(Markdown(summary_response.choices[0].message.content.strip()))

---
## Grasshopper bridge (reference)

Paste this into a **C# Script** component in Grasshopper.  
It watches the JSON file and updates output parameters whenever the council writes new weights.

In [ ]:
import json
import os

# Input: json_path (string) — full path to weights_output.json
# Right-click component → add inputs: json_path (str)
# Right-click component → add outputs: cost_gbp, carbon_eq, glazing_ratio, is_updated (bool)

if not json_path or not os.path.exists(json_path):
    cost_gbp      = 0.0
    carbon_eq     = 0.0
    glazing_ratio = 0.0
    is_updated    = False
else:
    with open(json_path, "r") as f:
        data = json.load(f)

    w = data["objective_weights"]

    cost_gbp      = w.get("cost_gbp",      0.0)
    carbon_eq     = w.get("carbon_eq",     0.0)
    glazing_ratio = w.get("glazing_ratio", 0.0)
    is_updated    = True